# Feature Engineering on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on creating new spark dataframes about the demand in different time series for HVFHV dataset and subsampling for plotting.

Since as a driver, they concentrate on where I go could increase the revenue, so we are focusing on the pickup location in the analysis of demand.

----

# Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col, count
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_hvfhv_demand")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/22 17:06:06 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.13.253.67 instead (on interface en0)
24/08/22 17:06:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/22 17:06:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read Files:

In [3]:
base_dir = "../data"
hvfhv_path = base_dir + '/developed/merged_data/full_hvfhv'
hvfhv_sdf = spark.read.parquet(hvfhv_path)

In [4]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "day_type",
    when(hvfhv_sdf["day_of_week"].isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]), "Weekday")
    .otherwise("Weekend")
)

hvfhv_sdf.show(5)

+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+-----------+-------------------------+-------------------+-----------------+------------------+-----------+-----------+------------+------------+-----+--------+
|hvfhs_license_num|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|wav_request_flag|wav_match_flag|day_of_week|request_to_pickup_minutes|         trip_speed|total_fare_amount|     total_revenue|pickup_date|pickup_hour|dropoff_date|dropoff_hour|month|day_type|
+-----------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+----------------+--------------+

In [5]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 100725779
Number of columns: 28


In [6]:
hvfhv_sdf.printSchema()

root
 |-- hvfhs_license_num: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: double (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: integer (nullable = true)
 |-- shared_match_flag: integer (nullable = true)
 |-- wav_request_flag: integer (nullable = true)
 |-- wav_match_flag: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- request_to_pickup_minutes: double (nullable = true)
 |-- trip_speed: double (nullable = true)
 |-- total_fare_amount: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 

In [8]:
TOTAL_HOURS = 24
TOTAL_DAYS = 31 + 31 + 30 + 31 + 30 + 31
TOTAL_MONTHS = 6 # the timeline is 6 month
TOTAL_LOCATIONS = hvfhv_sdf.select('PULocationID').distinct().count()

# Hourly Demand:

In [13]:
# Group by `pickup_hour`, `pickup_date``, `day_type`,
# then count the number of records for each group
hourly_pickup_demand_sdf = hvfhv_sdf.groupBy("pickup_hour", "pickup_date", "day_type") \
                                    .count() \
                                    .withColumnRenamed("count", "hourly_demand")
                                    
# Standardize the hourly revenue
hourly_pickup_demand_sdf = hourly_pickup_demand_sdf.withColumn('mean_hourly_demand',
                                                               col('hourly_demand') / TOTAL_HOURS)

hourly_pickup_demand_sdf = hourly_pickup_demand_sdf.orderBy("pickup_hour")\
                                                   .drop("hourly_demand")

hourly_pickup_demand_sdf.show(5)

+-----------+-----------+--------+------------------+
|pickup_hour|pickup_date|day_type|mean_hourly_demand|
+-----------+-----------+--------+------------------+
|          0| 2023-08-19| Weekend|          1261.625|
|          0| 2023-08-24| Weekday| 625.2083333333334|
|          0| 2023-08-27| Weekend|1409.8333333333333|
|          0| 2023-08-20| Weekend|1401.1666666666667|
|          0| 2023-08-26| Weekend|1221.0416666666667|
+-----------+-----------+--------+------------------+
only showing top 5 rows



In [25]:
hourly_pickup_demand_sdf.describe().show()

+-------+-----------------+--------+------------------+
|summary|      pickup_hour|day_type|mean_hourly_demand|
+-------+-----------------+--------+------------------+
|  count|             4416|    4416|              4416|
|   mean|             11.5|    NULL| 950.3866527022933|
| stddev|6.922970447632971|    NULL| 376.4472723475468|
|    min|                0| Weekday|             146.5|
|    max|               23| Weekend|          2088.625|
+-------+-----------------+--------+------------------+



# Hourly Demand Among Day of Week:

In [26]:
# Group by `day_of_week`, "day_type", "pickup_date", "pickup_hour"
# then count the number of records for each group
hourly_demand_among_day_of_week = hvfhv_sdf.groupBy("pickup_date", "day_of_week",
                                                    "day_type", "pickup_hour") \
                                            .count() \
                                            .orderBy("day_of_week") \
                                            .withColumnRenamed("count",
                                                               "day_by_hour_demand")

# Standardize the hourly revenue
hourly_demand_among_day_of_week = hourly_demand_among_day_of_week.withColumn('day_by_hour_demand',
                                                                             col('day_by_hour_demand') / TOTAL_HOURS)

hourly_demand_among_day_of_week.show(5)

+-----------+-----------+--------+-----------+------------------+
|pickup_date|day_of_week|day_type|pickup_hour|day_by_hour_demand|
+-----------+-----------+--------+-----------+------------------+
| 2023-08-18|     Friday| Weekday|         18|1400.9583333333333|
| 2023-08-25|     Friday| Weekday|         21|            1269.5|
| 2023-08-18|     Friday| Weekday|         22|1436.8333333333333|
| 2023-08-25|     Friday| Weekday|         20|1307.7083333333333|
| 2023-08-25|     Friday| Weekday|          6| 707.5416666666666|
+-----------+-----------+--------+-----------+------------------+
only showing top 5 rows



In [27]:
hourly_demand_among_day_of_week.describe().show()

+-------+-----------+--------+-----------------+------------------+
|summary|day_of_week|day_type|      pickup_hour|day_by_hour_demand|
+-------+-----------+--------+-----------------+------------------+
|  count|       4416|    4416|             4416|              4416|
|   mean|       NULL|    NULL|             11.5|  950.386652702292|
| stddev|       NULL|    NULL|6.922970447632987|376.44727234754635|
|    min|     Friday| Weekday|                0|             146.5|
|    max|  Wednesday| Weekend|               23|          2088.625|
+-------+-----------+--------+-----------------+------------------+



# Daily Demand:

### Pickup:

In [28]:
# Group by `pickup_date` and `PULocationID`, then count the number of records for each group
daily_pickup_demand_sdf = hvfhv_sdf.groupBy("pickup_date", "PULocationID") \
                                    .count() \
                                    .orderBy("pickup_date", "PULocationID") \
                                    .withColumnRenamed("count", "daily_demand")

# Standardize the daily revenue
daily_pickup_demand_sdf = daily_pickup_demand_sdf.withColumn('daily_demand', 
                                                             col('daily_demand') / TOTAL_DAYS)

daily_pickup_demand_sdf.show(5)

+-----------+------------+--------------------+
|pickup_date|PULocationID|        daily_demand|
+-----------+------------+--------------------+
| 2023-07-01|           2|0.016304347826086956|
| 2023-07-01|           3|   6.190217391304348|
| 2023-07-01|           4|  12.043478260869565|
| 2023-07-01|           5|  0.9836956521739131|
| 2023-07-01|           6|   1.815217391304348|
+-----------+------------+--------------------+
only showing top 5 rows



In [29]:
daily_pickup_demand_sdf.describe().show()

+-------+-----------------+--------------------+
|summary|     PULocationID|        daily_demand|
+-------+-----------------+--------------------+
|  count|            47387|               47387|
|   mean|132.8339206955494|   11.55217067880468|
| stddev|75.97943748101534|  10.465121493228825|
|    min|                1|0.005434782608695652|
|    max|              263|  109.22282608695652|
+-------+-----------------+--------------------+



### Dropoff:

In [30]:
# Group by `pickup_date` and `PULocationID`, then count the number of records for each group
daily_dropoff_demand_sdf = hvfhv_sdf.groupBy("pickup_date", "DOLocationID") \
                                    .count() \
                                    .orderBy("pickup_date", "DOLocationID") \
                                    .withColumnRenamed("count", "daily_demand")
# Standardize the daily revenue
daily_dropoff_demand_sdf = daily_dropoff_demand_sdf.withColumn('daily_demand', 
                                                               col('daily_demand') / TOTAL_DAYS)

daily_dropoff_demand_sdf.show(5)

+-----------+------------+--------------------+
|pickup_date|DOLocationID|        daily_demand|
+-----------+------------+--------------------+
| 2023-07-01|           1|  25.608695652173914|
| 2023-07-01|           2|0.021739130434782608|
| 2023-07-01|           3|   6.206521739130435|
| 2023-07-01|           4|   9.548913043478262|
| 2023-07-01|           5|  0.9184782608695652|
+-----------+------------+--------------------+
only showing top 5 rows



In [31]:
daily_dropoff_demand_sdf.describe().show()

+-------+-----------------+--------------------+
|summary|     DOLocationID|        daily_demand|
+-------+-----------------+--------------------+
|  count|            47621|               47621|
|   mean|132.2308855336931|  11.495405639455692|
| stddev|76.21439928249457|  11.091384473363675|
|    min|                1|0.005434782608695652|
|    max|              263|  116.23913043478261|
+-------+-----------------+--------------------+



# Monthly Demand:

In [9]:
# Group by `month`, 
# then count the number of records for each group
monthly_pickup_demand_sdf = hvfhv_sdf.groupBy("month") \
                                     .count() \
                                     .orderBy("month") \
                                     .withColumnRenamed("count", "monthly_demand")

# Standardize the monthly revenue
monthly_pickup_demand_sdf = monthly_pickup_demand_sdf.withColumn('monthly_demand', 
                                                                 col('monthly_demand') / TOTAL_MONTHS)

monthly_pickup_demand_sdf.show(5)

+-----+------------------+
|month|    monthly_demand|
+-----+------------------+
|    7|         2748802.5|
|    8|2640568.3333333335|
|    9|2835554.8333333335|
|   10|2895855.6666666665|
|   11|         2752377.5|
+-----+------------------+
only showing top 5 rows



In [10]:
monthly_pickup_demand_sdf.describe().show()

+-------+------------------+------------------+
|summary|             month|    monthly_demand|
+-------+------------------+------------------+
|  count|                 6|                 6|
|   mean|               9.5| 2797938.305555556|
| stddev|1.8708286933869707|  103767.147201019|
|    min|                 7|2640568.3333333335|
|    max|                12|         2914471.0|
+-------+------------------+------------------+



# Save the Merged Datasets:

Save the hourly pickup demand dataset:

In [34]:
hour_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_pickup_demand'
hour_path = os.path.join(hour_dir, file_name)
hourly_pickup_demand_sdf.write.mode('overwrite').parquet(hour_path)

Save hourly demand among day of week dataset:

In [35]:
week_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_demand_among_day_of_week'
week_path = os.path.join(week_dir, file_name)
hourly_demand_among_day_of_week.write.mode('overwrite').parquet(week_path)

Save the daily pickup demand dataset:

In [36]:
day_dir = base_dir + '/developed/merged_data'
file_name = 'daily_pickup_demand'
day_path = os.path.join(day_dir, file_name)
daily_pickup_demand_sdf.write.mode('overwrite').parquet(day_path)

Save the daily dropoff demand dataset:

In [37]:
day_dir = base_dir + '/developed/merged_data'
file_name = 'daily_dropoff_demand'
day_path = os.path.join(day_dir, file_name)
daily_dropoff_demand_sdf.write.mode('overwrite').parquet(day_path)

Save the monthly pickup demand dataset:

In [11]:
month_dir = base_dir + '/developed/merged_data'
file_name = 'monthly_pickup_demand'
month_path = os.path.join(month_dir, file_name)
monthly_pickup_demand_sdf.write.mode('overwrite').parquet(month_path)